# YOLOv8 Fine-tuning (상품 객체 인식)

**클래스:** bag, sunglasses, food_drink, shoes, clothing

**사전 준비:** `preprocess_data.py`로 생성된 `yolo_dataset/` 폴더를 zip으로 압축하여 Google Drive에 업로드

## 1. 환경 설정

In [ ]:
!pip install ultralytics

import torch
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 2. Google Drive 마운트 & 데이터 압축 해제

In [ ]:
import zipfile
import yaml
from google.colab import drive

drive.mount('/content/drive')

# Drive에 업로드한 zip 파일 경로 (필요시 수정)
ZIP_PATH = "/content/drive/MyDrive/yolo_dataset.zip"
# 압축 해제할 경로
EXTRACT_DIR = "/content"
DATASET_PATH = f"{EXTRACT_DIR}/yolo_dataset"

# zip 압축 해제
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)
print(f"압축 해제 완료: {ZIP_PATH} -> {DATASET_PATH}")

# data.yaml의 path를 절대 경로로 갱신
yaml_path = f"{DATASET_PATH}/data.yaml"
with open(yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["path"] = DATASET_PATH
with open(yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"데이터셋 경로: {DATASET_PATH}")
print(f"data.yaml: {data_cfg}")

## 3. 데이터 검증

In [ ]:
import os
from collections import Counter

# 이미지/라벨 수 확인
for split in ["train", "val"]:
    img_count = len(os.listdir(f"{DATASET_PATH}/images/{split}"))
    lbl_count = len(os.listdir(f"{DATASET_PATH}/labels/{split}"))
    print(f"{split}: 이미지 {img_count}장, 라벨 {lbl_count}개")

# 클래스 분포 확인
class_counter = Counter()
for split in ["train", "val"]:
    label_dir = f"{DATASET_PATH}/labels/{split}"
    for lbl_file in os.listdir(label_dir):
        with open(f"{label_dir}/{lbl_file}") as f:
            for line in f:
                cls_id = int(line.strip().split()[0])
                class_counter[cls_id] += 1

names = ["bag", "sunglasses", "food_drink", "shoes", "clothing"]
print("\n클래스 분포:")
for cid, name in enumerate(names):
    print(f"  {cid} ({name}): {class_counter.get(cid, 0)}개")

## 4. YOLOv8 Fine-tuning

- `yolov8n.pt` (nano): 가볍고 빠름, 서빙에 적합
- 데이터가 충분하므로 `yolov8s.pt` (small)로 변경 가능

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # pretrained 모델 로드

results = model.train(
    data=yaml_path,
    epochs=50,          # 필요시 조정
    imgsz=320,          # 원본이 300x300이므로 320 사용
    batch=32,           # GPU 메모리에 따라 조정 (T4: 32, A100: 64+)
    device=0,
    patience=10,        # 10 epoch 동안 개선 없으면 조기 종료
    save=True,
    project=f"{DATASET_PATH}/runs",
    name="product_detector",
)

## 5. 학습 결과 확인

In [ ]:
from IPython.display import Image, display

# 학습 곡선
results_dir = f"{DATASET_PATH}/runs/product_detector"
display(Image(f"{results_dir}/results.png", width=800))

# confusion matrix
display(Image(f"{results_dir}/confusion_matrix.png", width=600))

## 6. 검증 데이터로 테스트

In [ ]:
# best 모델로 검증
best_model = YOLO(f"{results_dir}/weights/best.pt")
metrics = best_model.val(data=yaml_path)

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 7. 샘플 추론 테스트

In [ ]:
import glob
import random
import cv2
import matplotlib.pyplot as plt

# val 이미지에서 랜덤 샘플 추론
val_images = glob.glob(f"{DATASET_PATH}/images/val/*.jpg")
samples = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, samples):
    results = best_model.predict(img_path, conf=0.5, verbose=False)
    annotated = results[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. 모델 다운로드

학습된 `best.pt`를 다운로드하여 FastAPI 프로젝트에 배치

In [ ]:
from google.colab import files

best_pt_path = f"{results_dir}/weights/best.pt"
print(f"모델 경로: {best_pt_path}")
print(f"모델 크기: {os.path.getsize(best_pt_path) / 1024 / 1024:.1f} MB")

# 로컬로 다운로드
files.download(best_pt_path)